# Stratified Downsampling to 5k cells / sample

**Input:** `atlas_concatenated_v2_subset_UC_inflamed_baseline_v2_filter_1000.h5ad`

**Rule:** For every `sample_id`, keep at most `TARGET_CELLS = 5000` cells while preserving the per-sample `cell_type` proportions. Samples with fewer than 5k cells are kept **as-is** (no discarding).

**Output:** `atlas_concatenated_v2_subset_UC_inflamed_baseline_v2_filter_1000_strat5k.h5ad`

In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc

INPUT_PATH   = "atlas_concatenated_v2_subset_UC_inflamed_baseline_v2_filter_1000.h5ad"
OUTPUT_PATH  = "atlas_concatenated_v2_subset_UC_inflamed_baseline_v2_filter_1000_strat5k.h5ad"
DONOR_COL    = "sample_id"
CELLTYPE_COL = "cell_type"
TARGET_CELLS = 5000
RANDOM_STATE = 42

In [2]:
adata = sc.read_h5ad(INPUT_PATH)
print('Loaded:', adata.shape)
print('Samples:', adata.obs[DONOR_COL].nunique())
print('Cell types:', adata.obs[CELLTYPE_COL].nunique())

sizes = adata.obs[DONOR_COL].value_counts()
print(f"\nSamples >= {TARGET_CELLS}: {(sizes >= TARGET_CELLS).sum()}")
print(f"Samples <  {TARGET_CELLS}: {(sizes <  TARGET_CELLS).sum()}")

Loaded: (1133533, 4396)
Samples: 377
Cell types: 110

Samples >= 5000: 62
Samples <  5000: 315


In [3]:
def stratified_downsample_keep_small(
    adata: sc.AnnData,
    donor_col: str,
    celltype_col: str,
    target_cells: int,
    random_state: int = 42,
    verbose: bool = True,
) -> sc.AnnData:
    """Donor-wise stratified downsampling, preserving cell-type proportions.

    Donors with fewer than ``target_cells`` cells are kept in full (no discard).
    Donors with >= ``target_cells`` are sampled to exactly ``target_cells``
    while maintaining the original per-donor cell-type distribution.
    """
    rng = np.random.default_rng(random_state)
    donors = sorted(adata.obs[donor_col].unique())

    # Pre-extract obs arrays once for speed
    donor_arr = adata.obs[donor_col].values
    ct_arr    = adata.obs[celltype_col].values
    all_idx   = np.arange(adata.n_obs)

    keep_indices = []
    n_kept_full, n_downsampled = 0, 0

    for donor_id in donors:
        donor_mask = donor_arr == donor_id
        donor_idx  = all_idx[donor_mask]
        n_cells    = donor_idx.size

        if n_cells <= target_cells:
            keep_indices.append(donor_idx)
            n_kept_full += 1
            if verbose:
                print(f"  {donor_id}: {n_cells:,} -> {n_cells:,} (kept all)")
            continue

        # Stratify by cell-type proportions within this donor
        donor_ct = ct_arr[donor_idx]
        ct_series = pd.Series(donor_ct)
        ct_counts = ct_series.value_counts()
        target_per_type = (ct_counts / n_cells * target_cells).round().astype(int)

        # Fix rounding so quotas sum to exactly target_cells
        diff = target_cells - int(target_per_type.sum())
        if diff != 0:
            target_per_type[target_per_type.idxmax()] += diff

        selected = []
        for celltype, n_target in target_per_type.items():
            ct_idx = donor_idx[donor_ct == celltype]
            sample_n = int(min(n_target, len(ct_idx)))
            if sample_n > 0:
                selected.append(rng.choice(ct_idx, size=sample_n, replace=False))

        selected = np.concatenate(selected) if selected else np.array([], dtype=int)
        keep_indices.append(selected)
        n_downsampled += 1
        if verbose:
            print(f"  {donor_id}: {n_cells:,} -> {selected.size:,} (downsampled)")

    final_idx = np.sort(np.concatenate(keep_indices))
    out = adata[final_idx].copy()

    if verbose:
        print(f"\nKept-in-full: {n_kept_full} samples | Downsampled: {n_downsampled} samples")
        print(f"Total cells: {adata.n_obs:,} -> {out.n_obs:,}")
    return out

In [4]:
adata_ds = stratified_downsample_keep_small(
    adata,
    donor_col=DONOR_COL,
    celltype_col=CELLTYPE_COL,
    target_cells=TARGET_CELLS,
    random_state=RANDOM_STATE,
    verbose=False,
)
print('Downsampled shape:', adata_ds.shape)
print('Samples retained:', adata_ds.obs[DONOR_COL].nunique())

Downsampled shape: (1006407, 4396)
Samples retained: 377


In [5]:
# Sanity checks
post = adata_ds.obs[DONOR_COL].value_counts()
print('Per-sample cell counts (post-downsampling):')
print(post.describe())
print(f"\nMax cells per sample: {post.max()} (should be <= {TARGET_CELLS})")
assert post.max() <= TARGET_CELLS, 'Downsampling cap violated!'
assert adata_ds.obs[DONOR_COL].nunique() == adata.obs[DONOR_COL].nunique(), \
    'Some samples were dropped — should not happen with keep-small policy'

# Cell-type proportions preserved (spot-check on a downsampled donor)
large_sample = adata.obs[DONOR_COL].value_counts().idxmax()
before = adata.obs.loc[adata.obs[DONOR_COL] == large_sample, CELLTYPE_COL].value_counts(normalize=True)
after  = adata_ds.obs.loc[adata_ds.obs[DONOR_COL] == large_sample, CELLTYPE_COL].value_counts(normalize=True)
compare = pd.concat([before, after], axis=1, keys=['before', 'after']).fillna(0)
compare['abs_diff'] = (compare['before'] - compare['after']).abs()
print(f"\nCell-type proportions for largest sample ({large_sample}):")
compare.sort_values('before', ascending=False).head(10)

Per-sample cell counts (post-downsampling):
count     377.000000
mean     2669.514589
std      1552.565071
min       501.000000
25%      1263.000000
50%      2306.000000
75%      4160.000000
max      5000.000000
Name: count, dtype: float64

Max cells per sample: 5000 (should be <= 5000)

Cell-type proportions for largest sample (intestinalDev_3_6):


,before,after,abs_diff
cell_type,,,
Unknown,1.0,1.0,0.0
C1Qhi IL1Blo macro,0.0,0.0,0.0
Th22,0.0,0.0,0.0
CD4 HSPhi Treg,0.0,0.0,0.0
pDC,0.0,0.0,0.0
ABCA8pos WNT2Bpos FOShi fibroblast,0.0,0.0,0.0
ABCA8pos WNT2Bpos FOSlo fibroblast,0.0,0.0,0.0
Arterial endothelium,0.0,0.0,0.0
C3hi RSPO3pos fibroblast,0.0,0.0,0.0


In [6]:
adata_ds.write_h5ad(OUTPUT_PATH, compression='gzip')
print('Saved ->', OUTPUT_PATH)
print('Size MB:', round(os.path.getsize(OUTPUT_PATH) / 1e6, 1))

Saved -> atlas_concatenated_v2_subset_UC_inflamed_baseline_v2_filter_1000_strat5k.h5ad
Size MB: 778.1
